# Experiment 4: DNN with Optimization Algorithms and Regularization Techniques

**Course:** AML ZC417 – Introduction to Deep Learning
**Module Reference:** Modules 4, 5, 6 — Deep Feedforward Networks, Optimization, Regularization
**Duration:** 2 hours

---

## Aim
To compare the effect of different optimization algorithms (SGD, SGD with Momentum, RMSprop, Adam) on training convergence, and to apply and compare regularization techniques (L2 weight decay, Dropout, Early Stopping, Batch Normalization) to reduce overfitting in a Deep Neural Network.

## Learning Outcomes
By the end of this experiment, you will be able to:
1. Explain how different optimizers update weights differently, and compare their convergence behaviour.
2. Deliberately induce overfitting using an over-capacity network, and recognize it from training/validation curves.
3. Apply L2 weight regularization and explain its effect on the loss function and weights.
4. Apply Dropout and explain why it is disabled during inference/evaluation.
5. Apply Early Stopping to halt training at the point of best generalization.
6. Apply Batch Normalization and explain its effect on training stability.
7. Combine multiple regularization techniques and evaluate the combined effect on test performance.


## Part 0 — Setup and Dataset

We continue using the **MNIST handwritten digits dataset**, as in Experiment 3, so we can isolate the effect of optimizers and regularization without changing the underlying data or task.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras

print("TensorFlow version:", tf.__version__)
tf.random.set_seed(42)
np.random.seed(42)


In [ ]:
(x_train_full, y_train_full), (x_test, y_test) = keras.datasets.mnist.load_data()

x_train_full = x_train_full.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0
x_train_full = x_train_full.reshape(-1, 784)
x_test = x_test.reshape(-1, 784)

# NOTE: To make overfitting clearly visible within a short training time,
# we deliberately train on a SMALL subset of the data (5,000 samples).
# A small training set + a large network is a classic recipe for overfitting.
x_train, x_val = x_train_full[:5000], x_train_full[5000:6000]
y_train, y_val = y_train_full[:5000], y_train_full[5000:6000]

print("Train shape:", x_train.shape, y_train.shape)
print("Validation shape:", x_val.shape, y_val.shape)
print("Test shape:", x_test.shape, y_test.shape)


---
## Part 1 — Comparing Optimization Algorithms

An **optimizer** determines exactly how model weights are updated using the gradients computed by backpropagation. We compare four widely used optimizers on the *same* architecture and data:

| Optimizer | Key Idea |
|---|---|
| SGD (plain) | Updates weights using only the current gradient, scaled by the learning rate. |
| SGD + Momentum | Accumulates a moving average of past gradients to smooth out updates and accelerate convergence. |
| RMSprop | Scales the learning rate per-parameter using a moving average of squared gradients — helps with noisy/sparse gradients. |
| Adam | Combines momentum and RMSprop-style per-parameter scaling — usually a strong default. |


In [ ]:
def build_model(optimizer, hidden_layers=[128, 64]):
    model = keras.Sequential()
    model.add(keras.layers.Input(shape=(784,)))
    for units in hidden_layers:
        model.add(keras.layers.Dense(units, activation="relu"))
    model.add(keras.layers.Dense(10, activation="softmax"))
    model.compile(optimizer=optimizer, loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

optimizers = {
    "SGD":              keras.optimizers.SGD(learning_rate=0.01),
    "SGD + Momentum":   keras.optimizers.SGD(learning_rate=0.01, momentum=0.9),
    "RMSprop":          keras.optimizers.RMSprop(learning_rate=0.001),
    "Adam":             keras.optimizers.Adam(learning_rate=0.001),
}

optimizer_histories = {}
for name, opt in optimizers.items():
    print(f"\nTraining with optimizer: {name}")
    model = build_model(opt)
    h = model.fit(x_train, y_train, validation_data=(x_val, y_val),
                  epochs=20, batch_size=32, verbose=0)
    optimizer_histories[name] = h
    print(f"{name}: final val_accuracy = {h.history['val_accuracy'][-1]:.4f}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for name, h in optimizer_histories.items():
    axes[0].plot(h.history["loss"], label=name)
    axes[1].plot(h.history["val_accuracy"], label=name)

axes[0].set_title("Training Loss by Optimizer"); axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss"); axes[0].legend()
axes[1].set_title("Validation Accuracy by Optimizer"); axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Accuracy"); axes[1].legend()
plt.tight_layout()
plt.show()


**What to look for:** Plain SGD typically converges slowest. SGD with Momentum should converge noticeably faster than plain SGD. Adam and RMSprop usually converge fastest in early epochs, since they adapt the effective learning rate per parameter.


---
## Part 2 — Deliberately Inducing Overfitting

To study regularization meaningfully, we first need a model that actually **overfits**. We use a large, high-capacity network trained on our small 5,000-sample training set for many epochs — a deliberate recipe for memorizing the training data rather than generalizing.


In [ ]:
def build_large_model():
    model = keras.Sequential([
        keras.layers.Input(shape=(784,)),
        keras.layers.Dense(512, activation="relu"),
        keras.layers.Dense(512, activation="relu"),
        keras.layers.Dense(512, activation="relu"),
        keras.layers.Dense(10, activation="softmax"),
    ])
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

overfit_model = build_large_model()
history_overfit = overfit_model.fit(
    x_train, y_train,
    validation_data=(x_val, y_val),
    epochs=40,
    batch_size=32,
    verbose=0,
)
print("Final train accuracy:", history_overfit.history["accuracy"][-1])
print("Final val accuracy:  ", history_overfit.history["val_accuracy"][-1])


In [ ]:
def plot_overfit_curves(history, title):
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].plot(history.history["loss"], label="Train Loss")
    axes[0].plot(history.history["val_loss"], label="Val Loss")
    axes[0].set_title(f"{title} - Loss"); axes[0].set_xlabel("Epoch"); axes[0].legend()

    axes[1].plot(history.history["accuracy"], label="Train Accuracy")
    axes[1].plot(history.history["val_accuracy"], label="Val Accuracy")
    axes[1].set_title(f"{title} - Accuracy"); axes[1].set_xlabel("Epoch"); axes[1].legend()
    plt.tight_layout()
    plt.show()

plot_overfit_curves(history_overfit, "Overfitting Baseline (3x512, no regularization)")


**What to look for:** Training accuracy should climb close to 100% while validation accuracy plateaus (or even declines) well below it — and validation loss should start rising after some epoch even as training loss keeps falling. This growing gap between train and validation curves **is** overfitting.


---
## Part 3 — L2 Weight Regularization (Weight Decay)

**L2 regularization** adds a penalty term (λ × sum of squared weights) to the loss function, discouraging any single weight from growing too large. This keeps the model simpler and less able to memorize noise in the training data.


In [ ]:
def build_l2_model(l2_lambda=0.001):
    reg = keras.regularizers.l2(l2_lambda)
    model = keras.Sequential([
        keras.layers.Input(shape=(784,)),
        keras.layers.Dense(512, activation="relu", kernel_regularizer=reg),
        keras.layers.Dense(512, activation="relu", kernel_regularizer=reg),
        keras.layers.Dense(512, activation="relu", kernel_regularizer=reg),
        keras.layers.Dense(10, activation="softmax"),
    ])
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

l2_model = build_l2_model(l2_lambda=0.001)
history_l2 = l2_model.fit(x_train, y_train, validation_data=(x_val, y_val),
                           epochs=40, batch_size=32, verbose=0)

plot_overfit_curves(history_l2, "With L2 Regularization (lambda=0.001)")
print("Final train accuracy:", history_l2.history["accuracy"][-1])
print("Final val accuracy:  ", history_l2.history["val_accuracy"][-1])


---
## Part 4 — Dropout

**Dropout** randomly "switches off" a fraction of neurons during each training step, forcing the network to not rely too heavily on any single neuron. This acts like training many different sub-networks and averaging them, which improves generalization. Dropout is automatically disabled during evaluation/inference — Keras handles this switch for you.


In [ ]:
def build_dropout_model(dropout_rate=0.5):
    model = keras.Sequential([
        keras.layers.Input(shape=(784,)),
        keras.layers.Dense(512, activation="relu"),
        keras.layers.Dropout(dropout_rate),
        keras.layers.Dense(512, activation="relu"),
        keras.layers.Dropout(dropout_rate),
        keras.layers.Dense(512, activation="relu"),
        keras.layers.Dropout(dropout_rate),
        keras.layers.Dense(10, activation="softmax"),
    ])
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

dropout_model = build_dropout_model(dropout_rate=0.5)
history_dropout = dropout_model.fit(x_train, y_train, validation_data=(x_val, y_val),
                                     epochs=40, batch_size=32, verbose=0)

plot_overfit_curves(history_dropout, "With Dropout (rate=0.5)")
print("Final train accuracy:", history_dropout.history["accuracy"][-1])
print("Final val accuracy:  ", history_dropout.history["val_accuracy"][-1])


---
## Part 5 — Early Stopping

**Early Stopping** monitors validation loss during training and stops training once it stops improving for a set number of epochs (the "patience"), then optionally restores the weights from the best epoch. This directly prevents the later epochs of overfitting from ever being used.


In [ ]:
early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
)

es_model = build_large_model()
history_es = es_model.fit(
    x_train, y_train,
    validation_data=(x_val, y_val),
    epochs=40,
    batch_size=32,
    callbacks=[early_stop],
    verbose=0,
)

print(f"Training stopped after {len(history_es.history['loss'])} epochs (out of 40 requested).")
plot_overfit_curves(history_es, "With Early Stopping (patience=5)")


---
## Part 6 — Batch Normalization

**Batch Normalization** normalizes the activations of a layer (mean 0, variance 1) for each mini-batch, then applies a learnable scale and shift. This stabilizes and often speeds up training, and provides a mild regularizing effect since each mini-batch is normalized slightly differently.


In [ ]:
def build_batchnorm_model():
    model = keras.Sequential([
        keras.layers.Input(shape=(784,)),
        keras.layers.Dense(512),
        keras.layers.BatchNormalization(),
        keras.layers.Activation("relu"),
        keras.layers.Dense(512),
        keras.layers.BatchNormalization(),
        keras.layers.Activation("relu"),
        keras.layers.Dense(512),
        keras.layers.BatchNormalization(),
        keras.layers.Activation("relu"),
        keras.layers.Dense(10, activation="softmax"),
    ])
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

bn_model = build_batchnorm_model()
history_bn = bn_model.fit(x_train, y_train, validation_data=(x_val, y_val),
                           epochs=40, batch_size=32, verbose=0)

plot_overfit_curves(history_bn, "With Batch Normalization")
print("Final train accuracy:", history_bn.history["accuracy"][-1])
print("Final val accuracy:  ", history_bn.history["val_accuracy"][-1])


---
## Part 7 — Combining Regularization Techniques

Regularization techniques are not mutually exclusive. Here we combine L2 regularization, Dropout, Batch Normalization, and Early Stopping into a single model, and compare its generalization gap (train accuracy − validation accuracy) against the unregularized baseline from Part 2.


In [ ]:
def build_combined_model(l2_lambda=0.0005, dropout_rate=0.3):
    reg = keras.regularizers.l2(l2_lambda)
    model = keras.Sequential([
        keras.layers.Input(shape=(784,)),
        keras.layers.Dense(512, kernel_regularizer=reg),
        keras.layers.BatchNormalization(),
        keras.layers.Activation("relu"),
        keras.layers.Dropout(dropout_rate),

        keras.layers.Dense(512, kernel_regularizer=reg),
        keras.layers.BatchNormalization(),
        keras.layers.Activation("relu"),
        keras.layers.Dropout(dropout_rate),

        keras.layers.Dense(512, kernel_regularizer=reg),
        keras.layers.BatchNormalization(),
        keras.layers.Activation("relu"),
        keras.layers.Dropout(dropout_rate),

        keras.layers.Dense(10, activation="softmax"),
    ])
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

combined_model = build_combined_model()
history_combined = combined_model.fit(
    x_train, y_train,
    validation_data=(x_val, y_val),
    epochs=40,
    batch_size=32,
    callbacks=[keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)],
    verbose=0,
)

plot_overfit_curves(history_combined, "Combined Regularization (L2 + Dropout + BatchNorm + EarlyStopping)")


In [ ]:
# Compare the generalization gap (train_acc - val_acc) across all approaches
results = {
    "No regularization":         history_overfit,
    "L2 regularization":         history_l2,
    "Dropout":                   history_dropout,
    "Early Stopping":            history_es,
    "Batch Normalization":       history_bn,
    "Combined":                  history_combined,
}

print(f"{'Configuration':<25}{'Train Acc':>12}{'Val Acc':>12}{'Gap':>10}")
print("-" * 59)
for name, h in results.items():
    train_acc = h.history["accuracy"][-1]
    val_acc = h.history["val_accuracy"][-1]
    print(f"{name:<25}{train_acc:>12.4f}{val_acc:>12.4f}{(train_acc - val_acc):>10.4f}")


In [ ]:
# Bar chart comparing the generalization gap
names = list(results.keys())
gaps = [h.history["accuracy"][-1] - h.history["val_accuracy"][-1] for h in results.values()]

plt.figure(figsize=(9, 4))
plt.bar(names, gaps, color="teal")
plt.ylabel("Train Accuracy - Val Accuracy (Generalization Gap)")
plt.title("Generalization Gap Across Regularization Techniques")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()


**Key takeaway:** A smaller generalization gap indicates better generalization to unseen data — this is the whole point of regularization. Note that regularization is not free: overly strong regularization (too-high L2 lambda, too-high dropout rate) can *underfit*, reducing both training and validation accuracy. The goal is always to tune regularization strength, not just to add as much as possible.


---
## Part 8 — Final Evaluation on the Test Set

Evaluate the combined regularized model on the held-out test set (used only once).


In [ ]:
test_loss, test_acc = combined_model.evaluate(x_test, y_test, verbose=0)
print(f"Combined regularized model -- Test accuracy: {test_acc:.4f}, Test loss: {test_loss:.4f}")


---
## Part 9 — In-Lab Exercise (to be completed and shown to the instructor)

1. Re-run the optimizer comparison from Part 1 with the learning rate for **all four optimizers set to 0.01**. Does Adam still perform best? Explain what you observe in a markdown cell.
2. Try **three different L2 lambda values** (0.0001, 0.001, 0.01) on the large model from Part 2. Plot the resulting validation accuracy for each, and identify which value gives the best trade-off between underfitting and overfitting.
3. Try **three different Dropout rates** (0.2, 0.5, 0.7) on the large model. What happens to training accuracy as the dropout rate increases? Explain why.
4. Modify the Early Stopping callback to use `patience=2` instead of `patience=5`. Does the model stop earlier? Is the final validation accuracy better or worse than with `patience=5`? Explain your observation.
5. In 3–4 sentences, state which single regularization technique gave the **best** improvement in generalization gap in your experiments, and propose one additional technique (not covered in this experiment) that could further help — you may refer to your course textbook or lecture notes.


In [ ]:
# TODO 1: Optimizer comparison with lr=0.01 for all optimizers
# your code here


_TODO 1 (continued): Your explanation here._

In [ ]:
# TODO 2: L2 lambda sweep (0.0001, 0.001, 0.01)
# your code here


_TODO 2 (continued): Your explanation here._

In [ ]:
# TODO 3: Dropout rate sweep (0.2, 0.5, 0.7)
# your code here


_TODO 3 (continued): Your explanation here._

In [ ]:
# TODO 4: Early stopping with patience=2
# your code here


_TODO 4 (continued): Your explanation here._

_TODO 5: Your summary and proposal here._

---
## Post-Lab Viva Questions

1. What is the difference between plain SGD and SGD with Momentum?
2. Why do Adam and RMSprop often converge faster than plain SGD in early training?
3. What does it mean for a model to "overfit," and what specific signal in the loss/accuracy curves indicates it?
4. How does L2 regularization modify the loss function, and what effect does this have on the learned weights?
5. Why must Dropout be disabled during evaluation/inference, and how does the framework handle this automatically?
6. What does the "patience" parameter control in Early Stopping?
7. What problem does Batch Normalization address, and at what level (per-layer, per-batch, or per-sample) does it operate?
8. Can regularization ever hurt model performance? Under what circumstance?

---
*End of Experiment 4*
